# GBDT vs NN

Thin orchestration notebook for the default `NN` vs `GBDT` analysis.

In [3]:
from __future__ import annotations

import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_dir = notebook_dir.parent
repo_root = project_dir.parent

sys.path.insert(0, str(project_dir / "src"))
sys.path.insert(0, str(repo_root / "tabarena" / "tabarena"))

In [ ]:
from mfa import load_config, run_analysis

config = load_config(project_dir / "configs" / "config_4.yaml")
result = run_analysis(
    config,
    datasets=[
        "Bank_Customer_Churn",
        "diabetes",
        "miami_housing",
        "physiochemical_protein",
        "website_phishing",
    ],
)
result.analysis_table.head()

16:30:03 INFO mfa.pipeline: Starting analysis: comparisons=tabpfn_old_vs_tabicl_old, tabpfn_new_vs_tabicl_new; scope=5 selected dataset(s); unit=dataset; method_variant=default,tuned; n_jobs=32


16:30:03 INFO mfa.pipeline: Stage 1/5 raw results: cache hit (3645 rows, 5 dataset(s))
16:30:03 INFO mfa.pipeline: Stage 2/5 meta-features: trace enabled; metafeature caches remain active, so live per-split diagnostics appear only on cache misses
16:30:03 INFO mfa.pipeline: Stage 2/5 meta-features: cache hit (87 rows, 5 dataset(s))
16:30:03 INFO mfa.pipeline: Stage 3/5 pairwise gaps: cache hit (156 rows, 5 dataset(s))
16:30:03 INFO mfa.pipeline: Stage 4/5 analysis table: joining and aggregating at dataset level
/work/mherre/tabular-meta-feature-analysis/meta-feature-analysis/src/mfa/aggregation.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset_level = analysis.groupby(GROUP_COLUMNS, as_index=False, dropna=False).agg(**aggregations)
/work/mherre/

,dataset,comparison_name,group_a_name,group_b_name,group_a_label,group_b_label,expected_direction,n_splits,n,d,...,best_a_error,best_a_norm_error,best_b_error,best_b_norm_error,delta_raw,delta_raw_std,delta_raw_sem,delta_norm,delta_norm_std,delta_norm_sem
0,Bank_Customer_Churn,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,tabpfn-new,tabicl-new,NaN,9,6666.666667,10.0,...,0.127787,0.240931,0.128107,0.265143,-0.000320,0.001064,0.000355,-0.024211,0.094995,0.031665
1,diabetes,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,tabpfn-new,tabicl-new,NaN,30,512.000000,8.0,...,0.160379,0.845272,0.160789,0.870382,-0.000409,0.005135,0.000938,-0.025110,0.147614,0.026951
2,miami_housing,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,tabpfn-new,tabicl-new,NaN,9,9184.000000,15.0,...,76335.753181,0.020074,77682.598416,0.156452,-1346.845235,914.422651,304.807550,-0.136378,0.094991,0.031664
3,physiochemical_protein,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,tabpfn-new,tabicl-new,NaN,9,30486.666667,9.0,...,3.005262,0.221556,2.942654,0.113033,0.062608,0.011189,0.003730,0.108523,0.018969,0.006323
4,website_phishing,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,tabpfn-new,tabicl-new,NaN,30,902.000000,9.0,...,0.221970,0.176043,0.222513,0.191110,-0.000542,0.004971,0.000908,-0.015066,0.127525,0.023283


In [5]:
import pandas as pd

# -- Inspect what the result object contains --
print(f"config_hash:        {result.config_hash}")
print(f"comparison_name:    {result.comparison_name}")
print(f"analysis_table:     {result.analysis_table.shape}")
print(f"gap_table:          {result.gap_table.shape}")
print(f"metafeature_table:  {result.metafeature_table.shape}")
print(f"correlation_results: {len(result.correlation_results)} features tested")
print(
    f"correction_result:  {result.correction_result.method if result.correction_result else None}"
)
print(f"multivariate_result: {result.multivariate_result}")

config_hash:        098848319ee90cfd
comparison_name:    None
analysis_table:     (8, 155)
gap_table:          (156, 17)
metafeature_table:  (87, 140)
correlation_results: 274 features tested
correction_result:  bh
multivariate_result: None


## Correlation summary table

In [6]:
import numpy as np

# Build a comprehensive table from correlation + correction results
corr_df = pd.DataFrame([r.__dict__ for r in result.correlation_results])

if result.correction_result is not None:
    corr_df["p_value_adj"] = result.correction_result.adjusted_p_values
    corr_df["rejected"] = result.correction_result.rejected

# Add a significance star column for quick scanning
p_col = "p_value_adj" if "p_value_adj" in corr_df.columns else "p_value"
corr_df["sig"] = np.where(
    corr_df[p_col] < 0.001,
    "***",
    np.where(corr_df[p_col] < 0.01, "**", np.where(corr_df[p_col] < 0.05, "*", "")),
)

display_cols = [
    "predictor",
    "statistic",
    "ci_lower",
    "ci_upper",
    "p_value",
    *(["p_value_adj", "rejected"] if "p_value_adj" in corr_df.columns else []),
    "sig",
    "n_observations",
    "direction_confirmed",
]

corr_df[display_cols].sort_values("p_value")

,predictor,statistic,ci_lower,ci_upper,p_value,p_value_adj,rejected,sig,n_observations,direction_confirmed
78,pymfe__n4.mean,1.0,1.0,1.0,0.0,0.0,True,***,3,None
204,pymfe__f2.mean,1.0,1.0,1.0,0.0,0.0,True,***,3,None
216,pymfe__t1,-1.0,-1.0,-1.0,0.0,0.0,True,***,3,None
82,pymfe__t4,1.0,1.0,1.0,0.0,0.0,True,***,3,None
81,pymfe__t3,1.0,1.0,1.0,0.0,0.0,True,***,3,None
...,...,...,...,...,...,...,...,...,...,...
167,constant_feature_fraction,NaN,NaN,NaN,NaN,NaN,False,,5,None
186,pymfe__nr_norm,NaN,NaN,NaN,NaN,NaN,False,,4,None
192,pymfe__sd_ratio,NaN,NaN,NaN,NaN,NaN,False,,1,None
213,pymfe__n2.mean,NaN,NaN,NaN,NaN,NaN,False,,2,None


## Correlation scatter plots

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="ticks", context="talk")

analysis = result.analysis_table.copy()
predictors = corr_df["predictor"].tolist()
n_preds = len(predictors)
ncols = min(4, n_preds)
nrows = int(np.ceil(n_preds / ncols))

fig, axes = plt.subplots(
    nrows, ncols, figsize=(5.5 * ncols, 5 * nrows), sharey=True, squeeze=False
)

for idx, predictor in enumerate(predictors):
    ax = axes[idx // ncols][idx % ncols]
    plot_df = analysis[[predictor, "delta_norm"]].dropna()

    # Use log scale for features that are strictly positive and span orders of magnitude
    use_log = (plot_df[predictor] > 0).all() and (
        plot_df[predictor].max() / plot_df[predictor].min() > 10
    )

    sns.scatterplot(data=plot_df, x=predictor, y="delta_norm", s=35, alpha=0.85, ax=ax)
    if use_log:
        ax.set_xscale("log")

    ax.set_xlabel(predictor)
    if idx % ncols == 0:
        ax.set_ylabel(r"$\Delta_{\ell\ell}$ (NN − GBDT)")
    else:
        ax.set_ylabel("")

    # Annotate with correlation and adjusted p-value
    row = corr_df.loc[corr_df["predictor"] == predictor].iloc[0]
    p_display = row.get("p_value_adj", row["p_value"])
    ax.text(
        0.05,
        0.95,
        f"r = {row['statistic']:.3f}\np = {p_display:.3f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=12,
        bbox={"facecolor": "#d9d9d9", "edgecolor": "#9e9e9e", "alpha": 0.9},
    )

# Hide unused subplots
for idx in range(n_preds, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

sns.despine()
fig.tight_layout()
plt.show()